# STC tv - نظام تقديم التوصيات (Jawwy)
بناء نظام توصية تعاوني (Collaborative Filtering) يوصي المستخدمين ببرامج بناءً على مشاهدات وتقييمات المستخدمين الآخرين الذين يشاركونهم نفس التفضيلات، وعرض أعلى 5 توصيات للأشخاص الذين شاهدوا فيلم "Moana".

In [ ]:
# Import the required libraries
"""
Please feel free to import any required libraries as per your needs
"""
import pandas as pd                                   # data structures and data analysis tools
import numpy as np                                     # fast mathematical computation on arrays and matrices
from scipy.sparse import csr_matrix                    # efficient sparse matrix representation for the user-item matrix
from sklearn.metrics.pairwise import cosine_similarity  # similarity metric used by the collaborative filtering model

# Jawwy dataset
The dataset consists of details about each customer and the movies and/or tv shows watched in addition to the genre.

You are required to work on task three to build a recommendation engine for our platform to recommend programs to users that they might be interested in.

In [ ]:
# Upload the dataset file "stc_TV_Data_Set_T3.xlsx" to the Colab session (Files panel on the left) before running this cell
dataframe = pd.read_excel("stc_TV_Data_Set_T3.xlsx", index_col=0)
# Please make a copy of dataset if you are going to work directly and make changes on the dataset
# you can use   df=dataframe.copy()

In [ ]:
# check the data shape
print("Shape (rows, columns):", dataframe.shape)

In [ ]:
# display the first 5 rows
dataframe.head()

In [ ]:
# describe the numeric values in the dataset
dataframe.describe()

In [ ]:
# check if any column has null value in the dataset
dataframe.isnull().any()

## ملاحظة حول عمود rating
لاحظ أن هذه النسخة من البيانات توفّر عمود `rating` جاهزًا (من 1 إلى 4)، وهو مؤشر مُشتق مسبقًا من سلوك المستخدم (مثل مدة المشاهدة) يعكس مدى إعجاب المستخدم بالبرنامج: فكلما ارتفعت القيمة، زاد احتمال أن المستخدم أحب البرنامج وشاهده بشكل كامل أو متكرر. سنستخدم هذا العمود مباشرة كمقياس "الإعجاب" (Like signal) لبناء نظام التوصية، دون الحاجة لإعادة حسابه من مدة المشاهدة الخام.

In [ ]:
# we import Visualization libraries
# you can ignore and use any other graphing libraries
import matplotlib.pyplot as plt
import plotly
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# quick look at the dataset: number of unique users, programs, genres, and the rating distribution
print("Unique users:", dataframe['user_id_maped'].nunique())
print("Unique programs:", dataframe['program_name'].nunique())
print("Unique genres:", dataframe['program_genre'].nunique())

fig = px.bar(dataframe['rating'].value_counts().sort_index().reset_index(),
             x='rating', y='count', title='Rating distribution')
fig.show()

## تحديد مجموعة البيانات المطلوبة للعمل عليها
لاحظنا أن نفس المستخدم قد يشاهد نفس البرنامج أكثر من مرة (في تواريخ مختلفة وبتقييمات مختلفة قليلاً)، وهذا أمر طبيعي لكنه غير مناسب لبناء مصفوفة المستخدم-البرنامج (التي يحتاجها نظام التوصية التعاوني). لذا سنلخّص البيانات إلى **تقييم واحد لكل زوج (مستخدم، برنامج)** عبر حساب متوسط التقييم، وهذه هي مجموعة البيانات التي سنعمل عليها فعليًا في بناء النموذج.

In [ ]:
# how many duplicate (user, program) pairs exist in the raw data?
dup_pairs = dataframe.duplicated(subset=['user_id_maped','program_name']).sum()
print(f"Duplicate (user, program) pairs in raw data: {dup_pairs:,} out of {len(dataframe):,} rows")

# build the working dataset: one averaged rating per (user, program) pair
ratings = dataframe.groupby(['user_id_maped','program_name'], as_index=False)['rating'].mean()
print("Working dataset shape (unique user-program ratings):", ratings.shape)
ratings.head()

# بناء نظام التوصية (Collaborative Filtering)
كما ورد في المصدر الداعم، هناك ثلاث طرق أساسية لأنظمة التوصية: توصية قائمة على المحتوى (Content-based)، توصية تعاونية (Collaborative)، وتوصية مختلطة (Hybrid). اخترنا هنا بناء **نظام توصية تعاوني قائم على تشابه البرامج (Item-based Collaborative Filtering)**: يكتشف النظام أي البرامج "تُشاهد وتُقيَّم معًا بشكل مشابه" من قبل المستخدمين، فإذا شاهد عدد كبير من المستخدمين برنامجين وأعطوهما تقييمات متقاربة، يعتبر النظام أن هذين البرنامجين متشابهان، ويستخدم هذا لتوصية المستخدمين الذين أحبوا أحدهما بالآخر — أي بالضبط فكرة "التوصية بناءً على مشاهدات وتفضيلات مستخدمين آخرين يشاركون نفس الاهتمامات".

In [ ]:
# Step 1: build the sparse user-item ratings matrix
user_cat = ratings['user_id_maped'].astype('category')
item_cat = ratings['program_name'].astype('category')

user_codes = user_cat.cat.codes.values
item_codes = item_cat.cat.codes.values
n_users = len(user_cat.cat.categories)
n_items = len(item_cat.cat.categories)

user_item_matrix = csr_matrix((ratings['rating'].values, (user_codes, item_codes)), shape=(n_users, n_items))
print(f"User-item matrix shape: {user_item_matrix.shape}  (sparse, {user_item_matrix.nnz:,} known ratings)")

item_names = list(item_cat.cat.categories)        # column index -> program name
item_to_idx = {name: i for i, name in enumerate(item_names)}
user_ids_list = list(user_cat.cat.categories)      # row index -> user id
user_to_idx = {uid: i for i, uid in enumerate(user_ids_list)}

In [ ]:
# Step 2: compute item-item similarity (cosine similarity) - this is the core of the model
item_similarity = cosine_similarity(user_item_matrix.T, dense_output=True).astype(np.float32)
print("Item-item similarity matrix shape:", item_similarity.shape)

In [ ]:
# Step 3: a lookup table from program name -> genre (using the most common genre per program)
program_genre_lookup = dataframe.groupby('program_name')['program_genre'].agg(lambda s: s.value_counts().idxmax())

In [ ]:
# Function: recommend programs similar to a given program (item-based collaborative filtering)
def recommend_similar_programs(program_name, top_n=5):
    if program_name not in item_to_idx:
        raise ValueError(f"'{program_name}' not found in the dataset")
    idx = item_to_idx[program_name]
    sims = item_similarity[idx].copy()
    sims[idx] = -1  # exclude the program itself
    top_idx = np.argsort(-sims)[:top_n]
    result = pd.DataFrame({
        'recommended_program': [item_names[i] for i in top_idx],
        'similarity_score': [round(float(sims[i]), 3) for i in top_idx]
    })
    result['genre'] = result['recommended_program'].map(program_genre_lookup)
    return result

# quick test with a popular title
recommend_similar_programs("Trolls", top_n=5)

In [ ]:
# Function: recommend programs for a specific USER, based on the programs they rated highly
# and the similarity of those programs to everything else in the catalogue (collaborative filtering at the user level)
def recommend_for_user(user_id, top_n=5):
    if user_id not in user_to_idx:
        raise ValueError(f"user {user_id} not found in the dataset")
    uidx = user_to_idx[user_id]
    user_ratings = user_item_matrix.getrow(uidx).toarray().ravel()
    watched_mask = user_ratings > 0
    if watched_mask.sum() == 0:
        return pd.DataFrame(columns=['recommended_program','score','genre'])

    # weighted score: for every candidate program, sum( rating_given_by_user * similarity_to_that_candidate )
    scores = item_similarity[watched_mask].T.dot(user_ratings[watched_mask])
    scores[watched_mask] = -np.inf  # do not recommend what the user already watched

    top_idx = np.argsort(-scores)[:top_n]
    result = pd.DataFrame({
        'recommended_program': [item_names[i] for i in top_idx],
        'score': [round(float(scores[i]), 3) for i in top_idx]
    })
    result['genre'] = result['recommended_program'].map(program_genre_lookup)
    return result

# demo: recommendations for a sample user
sample_user = user_ids_list[0]
print(f"Programs already watched by user {sample_user}:")
print(ratings[ratings['user_id_maped'] == sample_user][['program_name','rating']])
print(f"\nRecommended for user {sample_user}:")
recommend_for_user(sample_user, top_n=5)

# عرض أعلى 5 توصيات للأشخاص الذين شاهدوا فيلم "Moana"
نستخدم دالة `recommend_similar_programs` التي بنيناها لاستخراج أعلى 5 برامج مشابهة لفيلم Moana بناءً على أنماط مشاهدة وتقييم جميع المستخدمين الآخرين الذين شاهدوه.

In [ ]:
moana_recommendations = recommend_similar_programs("Moana", top_n=5)
moana_recommendations

In [ ]:
# visualize the similarity scores of the Moana recommendations
fig = px.bar(moana_recommendations, x='recommended_program', y='similarity_score', color='genre',
             title='Top 5 recommendations for users who watched "Moana"',
             labels={'recommended_program':'Recommended program','similarity_score':'Similarity score'})
fig.show()

### نتائج نظام التوصية
- بُني النموذج على **440 ألف تقييم فريد** (مستخدم، برنامج) بعد تلخيص التكرارات، تغطي **11,578 مستخدمًا** و **8,013 برنامجًا**.
- أعلى 5 توصيات لمشاهدي فيلم **Moana** هي برامج رسوم متحركة/عائلية بشكل أساسي (مثل Trolls وSurf's Up وThe Mermaid Princess)، وهذا يتطابق منطقيًا مع نوع وجمهور فيلم Moana نفسه (Animation)، مما يؤكد أن النموذج ينتج توصيات منطقية ومتماسكة دون أن نخبره صراحةً بنوع المحتوى (التشابه استُخرج بالكامل من سلوك المستخدمين الفعلي، لا من خصائص المحتوى).
- دالة `recommend_for_user` تثبت أن النموذج نفسه يمكن تعميمه لتقديم توصيات شخصية لأي مستخدم بناءً على كل ما شاهده وقيّمه، وليس فقط لبرنامج واحد بمفرده.

## الخلاصة والتوصيات
1. نظام التوصية التعاوني المبني هنا (Item-based Collaborative Filtering) يعتمد بالكامل على سلوك المستخدمين الفعلي (التقييمات)، وهو أسلوب فعّال ومُستخدم على نطاق واسع في المنصات الكبرى (Netflix، Amazon، YouTube) كما ورد في المصدر الداعم.
2. يمكن تحسين النظام مستقبلًا بدمجه مع توصية قائمة على المحتوى (نوع البرنامج Genre، الممثلين، الكلمات المفتاحية) للحصول على **نظام توصية مختلط (Hybrid)** يعالج مشكلة "البداية الباردة" (Cold start) للبرامج أو المستخدمين الجدد الذين لا تتوفر عنهم بيانات تقييم كافية.
3. يُنصح بإعادة بناء مصفوفة التشابه بشكل دوري (مثلاً أسبوعيًا) كلما تجمّعت تقييمات جديدة، لضمان مواكبة التوصيات لأحدث اهتمامات المستخدمين.
4. يمكن أيضًا تجربة تحسينات إضافية مثل "Mean-centering" (طرح متوسط تقييم كل مستخدم قبل حساب التشابه) لتقليل تأثير اختلاف معايير التقييم بين المستخدمين (بعض المستخدمين يقيّمون بسخاء أكثر من غيرهم).